In [11]:
# Cell 1: Setup project root (fix path)
from pathlib import Path
import os, sys

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD =", os.getcwd())

CWD = D:\logistics_AI


In [12]:
# Cell 2: Config
PROCESS_CODE = "IMPORT_CUSTOMS_CLEARANCE"

DATA_DIR = "data/synth_optimal_3process_v1"
REGISTRY_DIR = "registry/synth_optimal_v1"
MODEL_DIR = "model/process_models"

N_ESTIMATORS = 300
CONTAMINATION = 0.06
RANDOM_STATE = 42

INCLUDE_CONTEXT_NUMERIC = False

In [13]:
# Cell 3: Sanity check input data
import pandas as pd
from pathlib import Path

events_path = Path(DATA_DIR) / "events.csv"
df = pd.read_csv(events_path)

print("process_code counts (top):")
print(df["process_code"].value_counts().head(20))

if PROCESS_CODE not in set(df["process_code"].unique()):
    print(f"⚠️ No rows for {PROCESS_CODE} in {events_path}")
    print("=> Bạn chưa có customs trong data train hiện tại. Có 2 hướng:")
    print("   (1) Skip customs train (bình thường)")
    print("   (2) Dùng bộ data synth 3-process (normalized_events_synth.csv) để train customs.")
else:
    print("✅ Found customs rows:", (df["process_code"] == PROCESS_CODE).sum())

process_code counts (top):
process_code
TRUCKING_DELIVERY_FLOW      348000
WAREHOUSE_FULFILLMENT       276000
IMPORT_CUSTOMS_CLEARANCE     68576
Name: count, dtype: int64
✅ Found customs rows: 68576


In [14]:
# Cell 4: Train process model (CUSTOMS)
from api.process_ai.train import train_process_model

summary = train_process_model(
    data_dir=DATA_DIR,
    registry_dir=REGISTRY_DIR,
    model_root_dir=MODEL_DIR,
    process_code=PROCESS_CODE,
    include_context_numeric=INCLUDE_CONTEXT_NUMERIC,
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
)

summary

{'process_code': 'IMPORT_CUSTOMS_CLEARANCE',
 'events_rows_used': 68576,
 'cases_used': 4286,
 'include_context_numeric': False,
 'n_estimators': 300,
 'contamination': 0.06,
 'random_state': 42,
 'artifacts_dir': 'model\\process_models\\IMPORT_CUSTOMS_CLEARANCE',
 'validation_warnings': [],
 'feature_report': {'cases': 4286,
  'dropped_cases': 0,
  'repeated_case_count': 0,
  'missing_step_cases': 0}}

In [15]:
# Cell 5: Verify artifacts saved
from pathlib import Path

art_dir = Path(MODEL_DIR) / PROCESS_CODE
print("artifacts_dir:", art_dir.resolve())
print("exists:", art_dir.exists())

sorted([p.name for p in art_dir.glob("*")])

artifacts_dir: D:\logistics_AI\model\process_models\IMPORT_CUSTOMS_CLEARANCE
exists: True


['baselines.json',
 'feature_schema.json',
 'model.pkl',
 'scaler.pkl',
 'score_quantiles.json',
 'train_summary.json']

In [16]:
# Cell 6: Quick inference smoke test (1 case)
import pandas as pd
from pathlib import Path
from api.process_ai.inference import load_process_artifacts, analyze_case

art = load_process_artifacts(
    process_code=PROCESS_CODE,
    model_root_dir=MODEL_DIR,
    registry_dir=REGISTRY_DIR,
)

df = pd.read_csv(Path(DATA_DIR) / "events.csv")
cid = df.loc[df["process_code"] == PROCESS_CODE, "case_id"].iloc[0]
one = df[(df["process_code"] == PROCESS_CODE) & (df["case_id"] == cid)].copy()

result = analyze_case(one, art, allow_unknown_steps=False)
result

{'ok': True,
 'process_code': 'IMPORT_CUSTOMS_CLEARANCE',
 'case_id': 'ORD_1923969372',
 'risk_score': 31.0,
 'raw_anomaly': 0.404765,
 'overall_severity': 'NORMAL',
 'top_steps': [{'step_code': 'STEP_013_CARGO_RELEASED',
   'duration_min': 679.9,
   'p95': 611.192,
   'deviation_factor': 1.112,
   'z_score': 2.309,
   'severity': 'WATCHLIST'},
  {'step_code': 'STEP_015_HANDOFF_TO_CARRIER',
   'duration_min': 35.05,
   'p95': 49.275,
   'deviation_factor': 0.711,
   'z_score': 0.878,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_007_PHYSICAL_INSPECTION_COMPLETE',
   'duration_min': 643.867,
   'p95': 1018.504,
   'deviation_factor': 0.632,
   'z_score': 0.625,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_010_PAYMENT_CONFIRMED',
   'duration_min': 236.3,
   'p95': 441.608,
   'deviation_factor': 0.535,
   'z_score': 0.204,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_001_DOC_RECEIVED',
   'duration_min': 24.983,
   'p95': 47.446,
   'deviation_factor': 0.527,
   'z_score': 0.161